### https://deepwiki.com/search/given-the-following-agent-clas_f40f044c-9b61-4adb-82d5-34c545333614?mode=fast

In [1]:
# {{git_clone}}
# apashea pymdp (inferactively-pymdp pip version 0.0.7.1 with additional custom functions)
!git clone https://github.com/apashea/pymdp.git
import sys
sys.path.append('/content/pymdp')

Cloning into 'pymdp'...
remote: Enumerating objects: 221, done.
remote: Counting objects: 100% (221/221), done.
remote: Compressing objects: 100% (217/217), done.
remote: Total 221 (delta 120), reused 38 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (221/221), 271.76 KiB | 2.23 MiB/s, done.
Resolving deltas: 100% (120/120), done.


In [2]:
# {{library_imports}}
# Library imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import pymdp
from pymdp.maths import softmax
from pymdp.agent import Agent
from pymdp import inference, utils, control, maths
import datetime
import copy
import math
import random
#from pymdp.maths import softmax, softmax_obj_arr, spm_dot, spm_wnorm, spm_MDP_G, spm_log_single, spm_log_obj_array
#from pymdp.control import get_expected_obs, calc_expected_utility, calc_states_info_gain, calc_pA_info_gain, calc_pB_info_gain
#from pymdp import utils
#import copy

from pymdp.pymdp_external_custom import custom_test_print_function, update_posterior_policies_full_info, infer_policies_info, select_highest_equivalent, infer_states_info
from pymdp.pymdp_external_monitoring import infer_states_info_monitoring
from pymdp.pymdp_external_custom import update_A_MMP_distributional, update_gamma, update_C_MMP_distributional, update_E
from pymdp.pymdp_external_custom import compute_current_qs_bma, sample_action_timestep_dependent, _sample_action_marginal_external, _sample_policy_external
from pymdp.pymdp_external_custom import compute_xn_bma, compute_spm_lfp, compute_spm_dopamine, plot_spm_lfp, plot_spm_dopamine
from pymdp.pymdp_external_custom import safe_literal_eval, plot_model, plot_C_modalities, plot_factors, plot_transitions, plot_agent_data_focused

In [3]:
import numpy as np
from pymdp import utils
from pymdp.agent import Agent
from pymdp.maths import softmax

# ===== CONSTANTS AND DIMENSIONS =====
num_obs = [2, 4]  # [o_action, o_payoff]
num_states = [4, 3, 2]  # [s_type, s_stance, s_own]
num_controls = [1, 2, 2]  # [NULL for s_type, pi_social for s_stance, pi_social for s_own]

# Observation labels with proper prefixes - ALL STRINGS
o_action_labels = ['cooperate', 'defect']
o_payoff_labels = ['payoff_-1', 'payoff_1', 'payoff_3', 'payoff_5']  # String labels, not numeric

# Hidden state labels with s_ prefix
s_type_labels = ['cooperator', 'reciprocator', 'exploiter', 'random']
s_stance_labels = ['trusting', 'neutral', 'hostile']
s_own_labels = ['cooperate', 'defect']

# Control labels
null_action_labels = ['NULL']  # For uncontrollable factors
u_social_labels = ['cooperate', 'defect']  # For controllable factors

# Numeric payoff values for preferences (mapped to string observation indices)
payoff_numeric_values = [-1, 1, 3, 5]

# ===== A MATRIX (LIKELIHOOD) =====
A = utils.obj_array(len(num_obs))

# A[0] = P(o_action | s_type, s_stance, s_own) - conditioned on ALL hidden state factors
cooperation_probs = np.array([
    [0.95, 0.80, 0.55],  # cooperator
    [0.90, 0.70, 0.30],  # reciprocator
    [0.70, 0.35, 0.10],  # exploiter
    [0.60, 0.50, 0.35]   # random
])

A[0] = np.zeros((num_obs[0], num_states[0], num_states[1], num_states[2]))
for s_type in range(num_states[0]):
    for s_stance in range(num_states[1]):
        for s_own in range(num_states[2]):
            coop_prob = cooperation_probs[s_type, s_stance]
            A[0][0, s_type, s_stance, s_own] = coop_prob  # P(cooperate | s_type, s_stance, s_own)
            A[0][1, s_type, s_stance, s_own] = 1 - coop_prob  # P(defect | s_type, s_stance, s_own)

# Normalize A[0] to ensure proper probability distribution
A[0] = utils.norm_dist(A[0])

# A[1] = P(o_payoff | s_type, s_stance, s_own) - conditioned on ALL hidden state factors in correct order
payoff_table = {
    (0, 0): 3,  # (C,C) = 3
    (0, 1): -1, # (C,D) = -1
    (1, 0): 5,  # (D,C) = 5
    (1, 1): 1   # (D,D) = 1
}

A[1] = np.zeros((num_obs[1], num_states[0], num_states[1], num_states[2]))
for s_type in range(num_states[0]):
    for s_stance in range(num_states[1]):
        for s_own in range(num_states[2]):
            # Get partner cooperation probability
            coop_prob = cooperation_probs[s_type, s_stance]

            # Calculate payoff probabilities - map numeric payoffs to string observation indices
            for payoff_idx, payoff_val in enumerate(payoff_numeric_values):
                prob = 0.0
                # Sum over partner actions
                for partner_action in range(2):  # 0=cooperate, 1=defect
                    partner_prob = coop_prob if partner_action == 0 else (1 - coop_prob)
                    payoff = payoff_table[(s_own, partner_action)]
                    if payoff == payoff_val:
                        prob += partner_prob
                A[1][payoff_idx, s_type, s_stance, s_own] = prob

# Normalize A[1] to ensure proper probability distribution
A[1] = utils.norm_dist(A[1])

# ===== B MATRIX (TRANSITION) =====
B = utils.obj_array(len(num_states))

# # B[0] = P(s_type,t+1 | s_type,t) - uncontrollable, so only NULL action
# B[0] = np.zeros((num_states[0], num_states[0], num_controls[0]))  # Only 1 action (NULL)
# for i in range(num_states[0]):
#     for j in range(num_states[0]):
#         if i == j:
#             B[0][i, j, 0] = 0.95  # stay same type
#         else:
#             B[0][i, j, 0] = 0.017  # switch to other type

# # Normalize B[0]
# B[0] = utils.norm_dist(B[0])

# B[0] = P(s_type,t+1 | s_type,t) - uncontrollable, so only NULL action
B[0] = np.zeros((num_states[0], num_states[0], num_controls[0]))  # Shape: (4, 4, 1)
for i in range(num_states[0]):
    for j in range(num_states[0]):
        if i == j:
            B[0][i, j, 0] = 0.95  # stay same type, only action 0 available
        else:
            B[0][i, j, 0] = 0.017  # switch to other type, only action 0 available

B[0] = utils.norm_dist(B[0])

# B[1] = P(s_stance,t+1 | s_stance,t, pi_social) - controllable
B[1] = np.zeros((num_states[1], num_states[1], num_controls[1]))

# For pi_social = cooperate (action 0)
cooperate_transitions = np.array([
    [0.90, 0.30, 0.05],  # from trust
    [0.10, 0.60, 0.35],  # from neutral
    [0.00, 0.10, 0.60]   # from hostile
])
B[1][:, :, 0] = cooperate_transitions.T

# For pi_social = defect (action 1)
defect_transitions = np.array([
    [0.10, 0.05, 0.02],  # from trust
    [0.50, 0.35, 0.18],  # from neutral
    [0.40, 0.60, 0.80]   # from hostile
])
B[1][:, :, 1] = defect_transitions.T

# Normalize B[1]
B[1] = utils.norm_dist(B[1])

# B[2] = P(s_own,t+1 | s_own,t, pi_social) - deterministic and controllable
B[2] = np.zeros((num_states[2], num_states[2], num_controls[2]))
for action in range(num_controls[2]):
    B[2][action, :, action] = 1.0  # new s_own = chosen action

# Normalize B[2] (already normalized but for consistency)
B[2] = utils.norm_dist(B[2])

# ===== C MATRIX (PREFERENCES) =====
C = utils.obj_array(len(num_obs))

# C[0] = preferences over o_action - no direct preference
C[0] = np.array([0.0, 0.0])

# C[1] = preferences over o_payoff - use numeric values for preferences
temperature = 1.0
payoff_prefs = softmax(np.array(payoff_numeric_values) / temperature)
C[1] = payoff_prefs

# ===== D MATRIX (INITIAL BELIEFS) =====
D = utils.obj_array(len(num_states))

# D[0] = initial beliefs about s_type - uniform
D[0] = np.array([0.25, 0.25, 0.25, 0.25])

# D[1] = initial beliefs about s_stance - near-neutral
D[1] = np.array([0.2, 0.6, 0.2])

# D[2] = initial beliefs about s_own - uniform
D[2] = np.array([0.5, 0.5])

# ===== E MATRIX (POLICY PRIOR) =====
# For single-step horizon: uniform over all 4 possible policies
E = np.array([0.25, 0.25, 0.25, 0.25])  # 4 policies, each with 0.25 prior

# ===== CONSTRUCT POLICIES =====
# Create explicit policies for ALL control factors (including uncontrollable s_type)
# Policy format: [action_for_s_type, action_for_s_stance, action_for_s_own]
# s_type: 0 = NULL (uncontrollable), s_stance: 0 = cooperate, 1 = defect, s_own: 0 = cooperate, 1 = defect
policies = [
    np.array([[0, 0, 0]]),  # [NULL, cooperate, cooperate]
    np.array([[0, 0, 1]]),  # [NULL, cooperate, defect]
    np.array([[0, 1, 0]]),  # [NULL, defect, cooperate]
    np.array([[0, 1, 1]])   # [NULL, defect, defect]
]

# ===== DIRICHLET PRIORS =====
pA = utils.dirichlet_like(A, scale = 1.0)   # Prior matrix Dirichlet distribution for A matrix (likelihood model)
pB = utils.dirichlet_like(B, scale = 1.0)   # Prior matrix Dirichlet distribution for B matrix (state transitions model)
pD = utils.dirichlet_like(D, scale = 1.0)   # Prior matrix Dirichlet distribution for D matrix (priors about initial states)

# ===== CREATE AGENT =====
trust_game_agent = Agent(A, B, C, D, E, pA=pA, pB=pB, pD=pD, gamma=1.0,
                             policies=policies, policy_len=1, inference_horizon=3, control_fac_idx=[1,2],
                             inference_algo="MMP", sampling_mode='full', policy_sep_prior=True,
                             save_belief_hist=True, use_BMA=False, action_selection="deterministic")

print("Trust Game POMDP Agent created successfully!")
print(f"Agent dimensions: obs={num_obs}, states={num_states}, controls={num_controls}")
print(f"Observation labels: {o_action_labels}, {o_payoff_labels}")
print(f"State labels: {s_type_labels}, {s_stance_labels}, {s_own_labels}")
print(f"Control labels: {null_action_labels}, {u_social_labels}")
print(f"Number of policies: {len(policies)}")
for i, policy in enumerate(policies):
    print(f"Policy {i}: {policy[0]} -> [{u_social_labels[policy[0][0]]}, {u_social_labels[policy[0][1]]}]")

Trust Game POMDP Agent created successfully!
Agent dimensions: obs=[2, 4], states=[4, 3, 2], controls=[1, 2, 2]
Observation labels: ['cooperate', 'defect'], ['payoff_-1', 'payoff_1', 'payoff_3', 'payoff_5']
State labels: ['cooperator', 'reciprocator', 'exploiter', 'random'], ['trusting', 'neutral', 'hostile'], ['cooperate', 'defect']
Control labels: ['NULL'], ['cooperate', 'defect']
Number of policies: 4
Policy 0: [0 0 0] -> [cooperate, cooperate]
Policy 1: [0 0 1] -> [cooperate, cooperate]
Policy 2: [0 1 0] -> [cooperate, defect]
Policy 3: [0 1 1] -> [cooperate, defect]


In [4]:
# ===== CREATE AGENT =====
trust_game_agent = Agent(A, B, C, D, E, pA=pA, pB=pB, pD=pD, gamma=1.0,
                             policies=policies, policy_len=1, inference_horizon=3, control_fac_idx=[1,2],
                             inference_algo="MMP", sampling_mode='full', policy_sep_prior=True,
                             save_belief_hist=True, use_BMA=False, action_selection="deterministic")


# Set learning on/off
learn_A = True
learn_B = True
learn_C = True
learn_D = True
learn_E = True
monitoring = True


# Hard-code a list of observations for the agent
obs_list = [
    [0,1]    # test observation: ["cooperate","payoff_1"]
]
for t in range(len(obs_list)):
  obs = obs_list[t]
  # hidden state inference
  qs, xn, vn = infer_states_info(trust_game_agent, obs)         # returns posterior beliefs about hidden states (qs[factor][timestep][policy]), intermediate posterior beliefs, and intermediate prediction errors

  # policy inference
  q_pi, neg_efe, expected_utility, info_gain = infer_policies_info(trust_game_agent)

  # sample action from inferred policies
  sampled_actions = sample_action_timestep_dependent(trust_game_agent)
  trust_action = u_social_labels[int(sampled_actions[1])]              # store the social action


  # compute Bayesian Model Average of beliefs for current timestep in agent's inference window
  qs_bma = compute_current_qs_bma(trust_game_agent, current_timestep_idx=None, save_history=True)



  if learn_A == True:
    # if monitoring==True:
    #   print(f"t={t} : update_A_MMP_distributional(my_agent, obs, distr_obs=False)")
    update_A_MMP_distributional(trust_game_agent,obs, distr_obs=False)

  if learn_B == True:
    # if monitoring==True:
    #   print(f"t={t} : my_agent.update_B(qs_prev_formatted)")

    # Temporarily store the formatted current beliefs
    original_qs = trust_game_agent.qs
    trust_game_agent.qs = trust_game_agent.qs_current_bma
    trust_game_agent.update_B(trust_game_agent.qs_prev_bma)
    trust_game_agent.qs = original_qs  # restore original structure
    # trust_game_agent.B[0] = B[0]  # Reset
    # trust_game_agent.B[1] = B[1]  # Reset
    # trust_game_agent.B[2] = B[2]  # Reset

  if learn_C == True:
    # if monitoring==True:
    #   print(f"t={t} : update_C_MMP_distributional(my_agent, obs, distr_obs=False, modalities_to_learn=[1,2])")
    update_C_MMP_distributional(trust_game_agent, obs, lr_pC=0.5, initial_scale=1.0, distr_obs=False, modalities_to_learn=[0])

  if learn_D == True:
    # if monitoring==True:
    #   print(f"t={t} : my_agent.curr_timestep = {my_agent.curr_timestep}")
    #   print(f"t={t} : my_agent.update_D()")
    trust_game_agent.update_D()
    # my_agent.D[0]=D[0]  # Reset to original priors
    # my_agent.D[1]=D[1]  # Reset to original priors
    # my_agent.D[2]=D[2]  # Reset to original priors

  if learn_E == True:
    # if monitoring==True:
    #   print(f"t={t} : update_E(my_agent, q_pi, lr_pE=1.0, initial_scale=1.0)")
    update_E(trust_game_agent, q_pi, lr_pE=0.5, initial_scale=1.0)

  # You can access the various updated parameters and measures as attributes within the game, for example:
  print(f"================================================")
  print(f"Results for timestep {t}:")
  print(f"------------------------------------------------")
  print(f"F = {trust_game_agent.F}")
  print(f"------------------------------------------------")
  print(f"G = {trust_game_agent.G}")
  print(f"------------------------------------------------")
  print(f"A[0] = {trust_game_agent.A[0]}")
  print(f"------------------------------------------------")
  print(f"qs_bma = {trust_game_agent.qs_current_bma}")    # Bayesian model average (posterior beliefs about hidden states averaged over all policies)

Results for timestep 0:
------------------------------------------------
F = [-2.76464729 -2.76464729 -2.76412105 -2.76412105]
------------------------------------------------
G = [-3.85232647 -3.461823   -3.83094839 -3.37401025]
------------------------------------------------
A[0] = [[[[0.95       0.95163787]
   [0.8        0.82296497]
   [0.55       0.56792854]]

  [[0.9        0.90381638]
   [0.7        0.73957846]
   [0.3        0.33245241]]

  [[0.7        0.71177971]
   [0.35       0.43798162]
   [0.1        0.14291867]]

  [[0.6        0.61930991]
   [0.5        0.58133961]
   [0.35       0.38803221]]]


 [[[0.05       0.04836213]
   [0.2        0.17703503]
   [0.45       0.43207146]]

  [[0.1        0.09618362]
   [0.3        0.26042154]
   [0.7        0.66754759]]

  [[0.3        0.28822029]
   [0.65       0.56201838]
   [0.9        0.85708133]]

  [[0.4        0.38069009]
   [0.5        0.41866039]
   [0.65       0.61196779]]]]
-----------------------------------------------

/content/pymdp/pymdp/utils.py:156: UserWarning: Input array is not an object array...                    Casting the input to an object array
  warnings.warn(
